In [ ]:
!pip install -qU langchain langchain-community langchain-google-genai langchain-chroma transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 59.3 MB/s eta 0:00:00
 

In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [ ]:
import pandas as pd

books = pd.read_csv("/content/drive/MyDrive/recommender_systems/LLM/books_cleaned.csv")

In [ ]:
books['categories'].value_counts().reset_index().query("count > 50")

,categories,count
0,Fiction,2111
1,Juvenile Fiction,390
2,Biography & Autobiography,311
3,History,207
4,Literary Criticism,124
5,Religion,117
6,Philosophy,117
7,Comics & Graphic Novels,116
8,Drama,86
9,Juvenile Nonfiction,57


In [ ]:
category_mapping = {'Fiction' : "Fiction",
 'Juvenile Fiction': "Children's Fiction",
 'Biography & Autobiography': "Nonfiction",
 'History': "Nonfiction",
 'Literary Criticism': "Nonfiction",
 'Philosophy': "Nonfiction",
 'Religion': "Nonfiction",
 'Comics & Graphic Novels': "Fiction",
 'Drama': "Fiction",
 'Juvenile Nonfiction': "Children's Nonfiction",
 'Science': "Nonfiction",
 'Poetry': "Fiction"}

books["simple_categories"] = books["categories"].map(category_mapping)

In [ ]:
books[~books['simple_categories'].isna()].shape

(3743, 14)

In [ ]:
from transformers import pipeline
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli", device_map='auto')

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [ ]:
seq = books.loc[books['simple_categories'] == "Nonfiction", "description"].reset_index(drop=True)[0]

In [ ]:
candidate_labels = ['Fiction', 'Nonfiction', "Children's Fiction", "Children's Nonfiction"]

results = classifier(seq, candidate_labels)
results

{'sequence': 'Oscar Wilde had one of literary history\'s most explosive love affairs with Lord Alfred "Bosie" Douglas. In 1895, Bosie\'s father, the Marquess of Queensberry, delivered a note to the Albemarle Club addressed to "Oscar Wilde posing as sodomite." With Bosie\'s encouragement, Wilde sued the Marquess for libel. He not only lost but he was tried twice for "gross indecency" and sent to prison with two years\' hard labor. With this publication of the uncensored trial transcripts, readers can for the first time in more than a century hear Wilde at his most articulate and brilliant. The Real Trial of Oscar Wilde documents an alarmingly swift fall from grace; it is also a supremely moving testament to the right to live, work, and love as one\'s heart dictates.',
 'labels': ['Nonfiction',
  "Children's Nonfiction",
  "Children's Fiction",
  'Fiction'],
 'scores': [0.4821670353412628,
  0.2544456124305725,
  0.19461090862751007,
  0.06877640634775162]}

In [ ]:
print(f"The predicted label is {results['labels'][0]} with probability {results['scores'][0]}")

The predicted label is Nonfiction with probability 0.4821670353412628


In [ ]:
def classify_description(sequence_to_classify):
    results = classifier(sequence_to_classify, candidate_labels)['labels'][0]
    return results


In [ ]:
classify_description("A Great Footabll player journey")

"Children's Nonfiction"

## Compare Results

In [ ]:
test_data = books[~books['simple_categories'].isna()]

In [ ]:
### TRUE Labels
actual_class = test_data['simple_categories']
actual_class

,simple_categories
0,Fiction
2,Fiction
8,Fiction
30,Children's Fiction
46,Fiction
...,...
5178,Fiction
5188,Fiction
5189,Fiction
5195,Nonfiction


In [ ]:
predicted_class = test_data['description'].map(classify_description)
predicted_class

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,description
0,Fiction
2,Fiction
8,Fiction
30,Fiction
46,Nonfiction
...,...
5178,Fiction
5188,Children's Nonfiction
5189,Fiction
5195,Nonfiction


In [ ]:
actual_class.values

array(['Fiction', 'Fiction', 'Fiction', ..., 'Fiction', 'Nonfiction',
       'Nonfiction'], dtype=object)

In [ ]:
predicted_class.values

array(['Fiction', 'Fiction', 'Fiction', ..., 'Fiction', 'Nonfiction',
       'Nonfiction'], dtype=object)

In [ ]:
results_df = pd.DataFrame(
    {
        "Actual": actual_class.values,
        "Predicted": predicted_class.values
    }
)

In [ ]:
import numpy as np

results_df['Compare'] = np.where(results_df['Actual'] == results_df['Predicted'], 1, 0)

In [ ]:
results_df

,Actual,Predicted,Compare
0,Fiction,Fiction,1
1,Fiction,Fiction,1
2,Fiction,Fiction,1
3,Children's Fiction,Fiction,0
4,Fiction,Nonfiction,0
...,...,...,...
3738,Fiction,Fiction,1
3739,Fiction,Children's Nonfiction,0
3740,Fiction,Fiction,1
3741,Nonfiction,Nonfiction,1


In [ ]:
### Accuracy

sum(results_df.Compare.values) / results_df.shape[0]

np.float64(0.6561581619022174)

In [ ]:
from tqdm.notebook import tqdm

isbns = []
predicted_cats = []

missing_cats = books.loc[books['simple_categories'].isna(), ["isbn13", 'description']].reset_index(drop=True)


for i in tqdm(range(len(missing_cats))):
    sequence = missing_cats['description'][i]
    result = classify_description(sequence)
    predicted_cats.append(result)
    isbns += [missing_cats['isbn13'][i]]



  0%|          | 0/1454 [00:00<?, ?it/s]

In [ ]:
pred_missing_df = pd.DataFrame(
    {
        "isbn13": isbns,
        "predicted_categories": predicted_cats
     }
)

In [ ]:
books = pd.merge(books, pred_missing_df, on='isbn13', how='left')

In [ ]:
books['simple_categories'] = np.where(
    books['simple_categories'].isna(), books['predicted_categories'], books['simple_categories']
)

books = books.drop(columns = ['predicted_categories'])

In [ ]:
books.isna().sum()

,0
isbn13,0
isbn10,0
title,0
authors,32
categories,30
thumbnail,166
description,0
published_year,0
average_rating,0
num_pages,0


In [ ]:
books.to_csv("/content/drive/MyDrive/recommender_systems/LLM/books_simple_categories.csv", index=False)